# RCCO controller comparison flow

Trains and verifies one RCCO-based behavior cloning controller (encoder + MLP, AL5D) for each encoder in `sp_types`, and compares them on the same held-out demonstrations. For each encoder the flow runs the same stages as Flow_RCCO_BehaviorCloning (train the sensor processing, train the controller, verify it); a final Compare_RCCO stage charts the per-field errors of all controllers.

All controllers are trained in this flow's own workspace; controllers trained by other flows are not reused. See robot_controller/DESIGN-BehaviorCloningFlow.md and data/expruns/DESIGN-Flows.md.

In [ ]:
import sys
sys.path.append("..")
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
from flow import setup_flow, run_flow, display_flow_report
from demonstration.demopack import import_demopack, group_chooser_sp_bc_standard
from robot_controller.rcco_flow import (
    flow_families, generate_controller_stages, generate_compare)

In [ ]:
flow_name = "RCCO-Compare-automove"
demopack_name = "automove-pack-01"
demonstration_cam = "dev2"
# touch-apple has rc-position-targets outside the AL5D limits, which the
# position conversion rejects (see DESIGN-BehaviorCloningFlow.md)
# flow_name = "RCCO-Compare-touch-apple"
# demopack_name = "touch-apple"
# demonstration_cam = "dev0"

# The encoders of the compared controllers: resnet50, vgg19, vae, vae_gan
sp_types = ["resnet50", "vgg19", "vae", "vae_gan"]

# The number of epochs used for training.
# -- low values for debugging the flow
# epochs_sp = {"resnet50": 2, "vgg19": 2, "vae": 2, "vae_gan": 2}
# epochs_warmup = 1
# epochs_end_to_end = 1
epochs_sp = {"resnet50": 100, "vgg19": 100, "vae": 300, "vae_gan": 300}
epochs_warmup = 20
epochs_end_to_end = 20

# Reuse each producer directory, creating it only when absent.
creation_style = "exist-ok"
# Move each existing producer directory to a timestamped backup, then start fresh.
# creation_style = "version"
# Delete each existing producer directory, then start fresh.
# creation_style = "discard-old"

## Set up the flow directory and import the demonstrations
All controllers use the same demonstration split, so they are compared on the same testing demonstrations.

In [ ]:
expruns_path, results_path, notebooks_path = setup_flow(
    flow_name, flow_families(sp_types))

demopack_path = pathlib.Path(Config()["demopacks_path"], demopack_name).expanduser()
selection = import_demopack(demopack_path, group_chooser_sp_bc_standard)
data = {group: [[demopack_name, demo, demonstration_cam] for demo in selection[group]]
        for group in ["sp_training", "sp_validation", "bc_training",
                      "bc_validation", "bc_testing"]}
for group, entries in data.items():
    print(f"{group}: {[entry[1] for entry in entries]}")

## Generate the exp/runs
One controller per encoder, then the comparison of their verification runs.

In [ ]:
entries = []
for sp_type in sp_types:
    entries += generate_controller_stages(
        sp_type, data, epochs_sp[sp_type], epochs_warmup, epochs_end_to_end,
        creation_style)
entries.append(generate_compare(
    sp_types, f"Encoder comparison on {demopack_name}", creation_style))
for entry in entries:
    print(entry["name"], entry["notebook"])

## Run the flow
Run the notebooks with papermill. To follow the execution inside a notebook, open its executed notebook through the link printed before each stage. A failing notebook stops the flow; the final cell reports which stages completed and re-raises the error.

In [ ]:
flow_error = None
try:
    run_flow(entries, expruns_path, results_path, notebooks_path)
except Exception as error:
    flow_error = error

In [ ]:
final_results_path = pathlib.Path(results_path, "robot_controller_compare", "_flow_compare")
report = display_flow_report(
    entries, results_path, notebooks_path, final_results_path,
    flow_error=flow_error,
    preview_files=sorted(final_results_path.glob("*.png")))
if flow_error is not None:
    raise flow_error
if not report["all-results-present"]:
    raise Exception("Flow finished without all expected stage results.")